In [1]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
pd.set_option('display.max_columns', 60)

In [3]:
!pip install tqdm

In [4]:
import torch, esm, pickle, os
from tqdm.auto import tqdm

device = "mps" if torch.backends.mps.is_available() else \
         ("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

print("Loading ESM-2 650M (first run downloads ~2.5 GB; later runs use cache) ...")
model, alphabet = esm.pretrained.esm2_t33_650M_UR50D()
batch_converter = alphabet.get_batch_converter()
model.eval()
model = model.to(device)
print("Loaded. Parameters:", sum(p.numel() for p in model.parameters())/1e6, "M")

seqs = pd.read_csv("../data/sequences.csv")
print(f"Embedding {len(seqs)} sequences ...")

MAX_LEN = 1022

@torch.no_grad()
def embed_one(seq):
    s = seq[:MAX_LEN]
    _, _, toks = batch_converter([("p", s)])
    toks = toks.to(device)
    out = model(toks, repr_layers=[33])
    rep = out["representations"][33][0, 1:-1]
    return rep.mean(dim=0).cpu().numpy().astype("float32")

embeddings = {}
for _, row in tqdm(seqs.iterrows(), total=len(seqs)):
    try:
        embeddings[row['acc']] = embed_one(row['sequence'])
    except Exception as e:
        print(f"  Failed for {row['acc']}: {type(e).__name__}: {e}")

accs = list(embeddings.keys())
X = np.stack([embeddings[a] for a in accs])
np.savez_compressed("../data/features_esm2.npz", accs=np.array(accs), X=X.astype("float32"))
print(f"\nSaved {X.shape[0]} embeddings of dim {X.shape[1]}")
print(f"Truncated proteins (length > {MAX_LEN}): {(seqs['length'] > MAX_LEN).sum()}")

/opt/miniconda3/envs/biol466/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: mps
Loading ESM-2 650M (first run downloads ~2.5 GB; later runs use cache) ...
Downloading: "https://dl.fbaipublicfiles.com/fair-esm/models/esm2_t33_650M_UR50D.pt" to /Users/sunavbajaj/.cache/torch/hub/checkpoints/esm2_t33_650M_UR50D.pt
Downloading: "https://dl.fbaipublicfiles.com/fair-esm/regression/esm2_t33_650M_UR50D-contact-regression.pt" to /Users/sunavbajaj/.cache/torch/hub/checkpoints/esm2_t33_650M_UR50D-contact-regression.pt
Loaded. Parameters: 651.043254 M
Embedding 1279 sequences ...


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1279/1279 [22:22<00:00,  1.05s/it]



Saved 1279 embeddings of dim 1280
Truncated proteins (length > 1022): 180


In [5]:
data = np.load("../data/features_esm2.npz", allow_pickle=True)
X, accs = data['X'], data['accs']
norms = np.linalg.norm(X, axis=1)
print("shape:", X.shape, "dtype:", X.dtype)
print("first acc:", accs[0])
print(f"L2 norms: min={norms.min():.2f}, max={norms.max():.2f}, mean={norms.mean():.2f}")
print(f"NaNs? {np.isnan(X).any()}; zero-norm vectors? {(norms == 0).sum()}")

shape: (1279, 1280) dtype: float32
first acc: A0A090N8E9
L2 norms: min=4.53, max=10.04, mean=7.23
NaNs? False; zero-norm vectors? 0
